# Import

In [1]:
import pandas as pd
import json

from curation_tools.curation_tools import (
    CuratedDataset,
    ObsSchema,
    VarSchema,
    Experiment,
    download_file,
    upload_parquet_to_bq
)

import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    handlers=[
        logging.FileHandler("curation.log"),
        logging.StreamHandler(),  # keep console output too
    ],
    force=True,
)
# show all columns
pd.set_option('display.max_columns', None)

# Download data

In [2]:
# !aws s3 cp --no-sign-request s3://genome-scale-tcell-perturb-seq/marson2025_data/D1_Rest.assigned_guide.h5ad ../non_curated/h5ad/zhu_2025_D1_rest_cl.h5ad
# !mv /content/PerturbationCatalogue/data_exploration/Perturbseq/non_curated/h5ad/D1_Rest.assigned_guide.h5ad /content/PerturbationCatalogue/data_exploration/Perturbseq/non_curated/h5ad/zhu_2025_pseudobulk.h5ad

In [3]:
!pwd

/nfs/production/mfreeberg/perturb-seq/aleks/PerturbationCatalogue/data_exploration/Perturbseq/curation_notebooks


# Initialise the dataset object

In [4]:
noncurated_path = '/hps/nobackup/mfreeberg/marson_downloads/D1_Rest.assigned_guide.h5ad'
cur_data = CuratedDataset(
    obs_schema=ObsSchema,
    var_schema=VarSchema,
    exp_metadata_schema=Experiment,
    noncurated_path=noncurated_path
)                    

cur_data.load_data()

Loading data from /hps/nobackup/mfreeberg/marson_downloads/D1_Rest.assigned_guide.h5ad


In [6]:
cur_data.adata.obs

,lane_id,n_genes_by_counts,total_counts,pct_counts_mt,top_guide_UMI_counts,guide_id,perturbed_gene_name,perturbed_gene_id,guide_type,PuroR,guide_group,low_quality
AAACAAGCAAACCGGTAACGGGAA-1_CD4i_R1L08_CD4i_R1_D1_Rest_CD4i_R1_Ultima,CD4i_R1L08,3945,8874.0,0.439486,169.0,multi_sgRNA,NaN,NaN,targeting,0.587712,multi sgRNA,False
AAACAAGCAAACCGGTATGTTGAC-1_CD4i_R1L08_CD4i_R1_D1_Rest_CD4i_R1_Ultima,CD4i_R1L08,1953,2969.0,0.269451,NaN,NaN,NaN,NaN,NaN,2.100727,no sgRNA,False
AAACAAGCAAAGGGATAGTAGGCT-1_CD4i_R1L08_CD4i_R1_D1_Rest_CD4i_R1_Ultima,CD4i_R1L08,2989,5807.0,0.654383,75.0,multi_sgRNA,NaN,NaN,targeting,2.826730,multi sgRNA,False
AAACAAGCAAATACCGATGTTGAC-1_CD4i_R1L08_CD4i_R1_D1_Rest_CD4i_R1_Ultima,CD4i_R1L08,4576,13363.0,1.017735,22.0,NTC-461,NTC,NTC,non-targeting,1.431987,targeting single sgRNA,False
AAACAAGCAAATCACGATGTTGAC-1_CD4i_R1L08_CD4i_R1_D1_Rest_CD4i_R1_Ultima,CD4i_R1L08,1890,3016.0,0.397878,9.0,PLA2G10-2,PLA2G10,ENSG00000069764,targeting,1.209989,targeting single sgRNA,False
...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGTGAGTTGTGACTAACGGGAA-1_CD4i_R1L21_CD4i_R1_D1_Rest_CD4i_R1_Ultima,CD4i_R1L21,2725,5182.0,0.135083,NaN,NaN,NaN,NaN,NaN,1.451924,no sgRNA,False
TTTGTGAGTTGTGACTATGTTGAC-1_CD4i_R1L21_CD4i_R1_D1_Rest_CD4i_R1_Ultima,CD4i_R1L21,4145,10171.0,0.304788,NaN,NaN,NaN,NaN,NaN,1.466352,no sgRNA,False
TTTGTGAGTTTAACCAAGTAGGCT-1_CD4i_R1L21_CD4i_R1_D1_Rest_CD4i_R1_Ultima,CD4i_R1L21,3504,9302.0,0.225758,NaN,NaN,NaN,NaN,NaN,0.000000,no sgRNA,False
TTTGTGAGTTTCGCCTACTTTAGG-1_CD4i_R1L21_CD4i_R1_D1_Rest_CD4i_R1_Ultima,CD4i_R1L21,2624,4919.0,0.203293,25.0,KIF18A-2,KIF18A,ENSG00000121621,targeting,1.492056,targeting single sgRNA,False


In [6]:
cur_data.adata.var

,gene_ids,feature_types,genome,gene_name,mt
ENSG00000000003,ENSG00000000003,Gene Expression,GRCh38,TSPAN6,False
ENSG00000000005,ENSG00000000005,Gene Expression,GRCh38,TNMD,False
ENSG00000000419,ENSG00000000419,Gene Expression,GRCh38,DPM1,False
ENSG00000000457,ENSG00000000457,Gene Expression,GRCh38,SCYL3,False
ENSG00000000460,ENSG00000000460,Gene Expression,GRCh38,C1orf112,False
...,...,...,...,...,...
ENSG00000291122,ENSG00000291122,Gene Expression,GRCh38,CASTOR3P,False
ENSG00000291135,ENSG00000291135,Gene Expression,GRCh38,FCGR1BP,False
ENSG00000291145,ENSG00000291145,Gene Expression,GRCh38,PPP5D1P,False
ENSG00000291237,ENSG00000291237,Gene Expression,GRCh38,SOD2,False


# OBS slot curation

### Since for multi-guide perturbations the identities of said guides are unknown, we are filtering out these cells.

In [7]:
print(f"Number of cells before filtering: {cur_data.adata.n_obs}")
cur_data.adata = cur_data.adata[cur_data.adata.obs['guide_id'] != 'multi_sgRNA']
print(f"Number of cells after filtering: {cur_data.adata.n_obs}")

Number of cells before filtering: 3074496
Number of cells after filtering: 2548393


### Add `index` as `perturbation_name`

In [8]:
cur_data.adata.obs['cell_barcode'] = cur_data.adata.obs.index.str.split('_').str[0]
cur_data.adata.obs['perturbation_name'] = cur_data.adata.obs['cell_barcode'] +'_'+cur_data.adata.obs['lane_id'].astype(str)

/tmp/ipykernel_2467654/2231587657.py:1: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  cur_data.adata.obs['cell_barcode'] = cur_data.adata.obs.index.str.split('_').str[0]


: 

### Show unique perturbations

In [ ]:
cur_data.show_unique(slot = 'obs', column = 'perturbation_name')

### Add guide RNA information

In [ ]:
# download the guide RNA spreadsheet
download_file(
    url="https://raw.githubusercontent.com/emdann/GWT_perturbseq_analysis_2025/refs/heads/master/metadata/suppl_tables/sgrna_library_metadata.suppl_table.csv",
    dest_path="../supplementary/zhu_2025_guide_info.csv"
)
# read in the guide RNA info csv
guide_info_df = pd.read_csv("../supplementary/zhu_2025_guide_info.csv")

guide_info_df = guide_info_df[['sgRNA', 'seq']]

guide_info_df['sgRNA'] = guide_info_df['sgRNA'].str.replace('1-Jun','JUN-1').str.replace('2-Jun','JUN-2')

guide_info_df = guide_info_df.rename(columns={'seq': 'guide_sequence', 'sgRNA':'guide_id'})

guide_info_df


In [ ]:
print("Number of overlapping guides with cur_data:")
cur_data.adata.obs['guide_id'].isin(guide_info_df['guide_id'].to_list()).value_counts()

In [ ]:
# merge cur_data.adata.obs with guide_info_df
cur_data.adata.obs = cur_data.adata.obs.merge(guide_info_df, on='guide_id', how='left')
# check that there are no missing guide sequences
print(f"Number of missing guide sequences: {cur_data.adata.obs['guide_sequence'].isna().sum()}")


In [ ]:
cur_data.adata.obs

In [ ]:
# change perturbed_gene_id from categorical to string
cur_data.adata.obs['perturbed_gene_id'] = cur_data.adata.obs['perturbed_gene_id'].astype(str)
cur_data.adata.obs['perturbed_gene_name'] = cur_data.adata.obs['perturbed_gene_name'].astype(str)

cur_data.adata.obs.loc[cur_data.adata.obs['perturbed_gene_id'].isin(['NTC']), 'perturbed_gene_id'] = 'control_nontargeting'
cur_data.adata.obs.loc[cur_data.adata.obs['perturbed_gene_name'].isin(['NTC']), 'perturbed_gene_name'] = 'control_nontargeting'

cur_data.adata.obs.loc[cur_data.adata.obs['perturbed_gene_id'].isna(), 'perturbed_gene_id'] = 'control_casonly'
cur_data.adata.obs.loc[cur_data.adata.obs['perturbed_gene_name'].isna(), 'perturbed_gene_name'] = 'control_casonly'

### Standardise perturbation targets

In [ ]:
cur_data.standardize_genes(
    slot='obs',
    input_column='perturbed_gene_id',
    input_column_type='ensembl_gene_id',
    multiple_entries=False,
    # remove_version=True,
    # version_sep='.'
)

In [ ]:
cur_data.adata.obs['perturbed_gene_id'][0]

# Manually replace some genes

In [ ]:
# define genes to replace in the cur_data.adata.obs
genes_to_replace = cur_data.adata.obs[cur_data.adata.obs['perturbed_target_symbol'].isna()]['perturbed_gene_name'].drop_duplicates().to_list()
# define columns to replace
cols_to_replace = ['perturbed_target_ensg',	'perturbed_target_symbol',	'perturbed_target_biotype',	'perturbed_target_coord',	'perturbed_target_chromosome']
# define mapping dict for renaming columns in gene_ont for replacement
gene_ont_col_mapping = {
    'ensembl_gene_id':'perturbed_target_ensg',
    'gene_symbol':'perturbed_target_symbol',
    'biotype':'perturbed_target_biotype',
    'gene_coord':'perturbed_target_coord',
    'chromosome_name':'perturbed_target_chromosome'
}
# loop through genes and fill in the missing cols
for replacement_gene in genes_to_replace:
    if replacement_gene in cur_data.gene_ont['synonym'].values:
        # subset gene_ont and get values for replacement
        replacement = (
            cur_data.gene_ont.rename(columns=gene_ont_col_mapping)
            .loc[cur_data.gene_ont['synonym'].isin([replacement_gene]), cols_to_replace]
        )[cols_to_replace].values[0]
        # replace values in cur_data.adata.obs with the values from gene_ont
        cur_data.adata.obs.loc[cur_data.adata.obs['perturbed_gene_name'].isin([replacement_gene]), cols_to_replace] = replacement
        print(f"Replaced {replacement_gene} with {replacement}")
    else:
        print(f"No replacement found for {replacement_gene}")


In [ ]:
# replace non-mapped perturbed_target_symbol with original gene symbols
cur_data.adata.obs.loc[cur_data.adata.obs['perturbed_target_symbol'].isna(), 'perturbed_target_symbol'] = cur_data.adata.obs.loc[cur_data.adata.obs['perturbed_target_symbol'].isna(), 'perturbed_gene_name'].values

In [ ]:
# replace entries in perturbed_target_ensg that are not ENSEMBL ids with NA
cur_data.adata.obs.loc[(~cur_data.adata.obs['perturbed_target_ensg'].str.startswith('ENSG')) & (cur_data.adata.obs['perturbed_target_ensg'] != 'control_nontargeting'), 'perturbed_target_ensg'] = pd.NA

In [ ]:
cur_data.adata.obs

### Add `perturbed_target_number` column

In [ ]:
# cur_data.count_entries(
#     slot='obs',
#     input_column='perturbed_target_ensg',
#     count_column_name='perturbed_target_number',
#     sep='|'
# )
cur_data.adata.obs['perturbed_target_number'] = 1

### Encode chromosomes as integers

In [ ]:
cur_data.chromosome_encoding()

In [ ]:
cur_data.adata.obs[['perturbation_name', 'perturbed_target_chromosome_encoding']]

In [ ]:
cur_data.adata.obs

### Curate replicates

In [ ]:
cur_data.adata.obs = cur_data.adata.obs.rename(columns={'lane_id':'technical_replicate'})

### Add metadata

In [ ]:
cur_data.create_columns(
    overwrite=True,
    slot="obs",
    col_dict={
        "dataset_id": cur_data.dataset_id,
        "sample_id": range(1, cur_data.adata.obs.shape[0] + 1),
        # perturbation type
        "perturbation_type_label": "CRISPRi",
        "perturbation_type_id": None,
        "data_modality": "Perturb-seq",
        "significant": None,
        "significance_criteria": None,
        "score_interpretation": None,

        # treatment
        "treatment_label": "untreated control",
        "treatment_id": "NCIT:C184729",
        # replicates
        # "technical_replicate": None,
        "biological_replicate": "D1_CE0008162",
        # model system
        "model_system_label": "primary_cell",
        "model_system_id": None,
        "tissue": "lymphoid tissue",
        "cell_line_label": None,
        "cell_line_id": None,
        "cell_type_label": "CD4-positive, alpha-beta T cell",
        "disease_label": "healthy",
        "disease_id": None,

        "timepoint": "P12DT8H0M0S",
        "species": "Homo sapiens",
        "sex_label": "female",
        "sex_id": None,
        "developmental_stage_label": "adult",
        "developmental_stage_id": None,

        "study_title": "Genome-scale perturb-seq in primary human CD4+ T cells maps context-specific regulators of T cell programs and human immune traits",
        "study_uri": "https://doi.org/10.64898/2025.12.23.696273",
        "study_year": 2025,
        "first_author": "Ronghui Zhu",
        "last_author": "Alexander Marson",

        "experiment_title": "Perturb-seq of primary human CD4-positive T cells for patient D1_CE0008162 under resting conditions",
        "experiment_summary": """
            Isolated human CD4-positive cells from four healthy donors were stimulated with ImmunoCult CD3/CD28/CD2 activator and sequentially transduced with dCas9-KRAB-Zim3 lentivirus (next morning after stimulation) and a Perturb-seq guide library (next afternoon after stimulation, MOI 0.2).
            The library consisted all genes expressed in human CD4+ T cells, all transcription factors annotated in the Lambert et al. (2018) and non-targeting controls totalling 12,748 genes.
            The gRNA sequences were selected from hCRISPRiv2 and Dolcetto libraries.
            On day 12, cells were split into three conditions:
            (1) Rest - cells were left for 8 hr without stimulation;
            (2) Stim8hr - cells were stimulated with ImmunoCult CD3/CD28/CD2 activator for 8 hr;
            (3) Stim48hr - cells were stimulated with ImmunoCult CD3/CD28/CD2 activator for 48 hr.
            Cells were harvested, fixed, stored using GEM-X Flex Sample Preparation v2 Kit and sequenced using GEM-X Flex Gene Expression Human n-plex kit converted into Ultima compatible libraries for sequencing on the Ultima Genomics UG100.
            """,

        "number_of_perturbed_targets": len(set(cur_data.adata.obs['perturbed_target_symbol'])),
        "number_of_perturbed_samples": cur_data.adata.obs.shape[0],

        "library_generation_type_id": "EFO:0022868",
        "library_generation_type_label": "endogenous",

        "library_generation_method_id": None,
        "library_generation_method_label": "dCas9-KRAB-Zim3",

        "enzyme_delivery_method_id": None,
        "enzyme_delivery_method_label": "lentivirus transduction",

        "library_delivery_method_id": None,
        "library_delivery_method_label": "lentivirus transduction",

        "enzyme_integration_state_id": None,
        "enzyme_integration_state_label": "random locus integration",

        "library_integration_state_id": None,
        "library_integration_state_label": "random locus integration",

        "enzyme_expression_control_id": None,
        "enzyme_expression_control_label": "constitutive transgene expression",

        "library_expression_control_id": None,
        "library_expression_control_label": "constitutive transgene expression",

        "library_name": "custom",
        "library_uri": None,

        "library_format_id": None,
        "library_format_label": "pooled",

        "library_scope_id": None,
        "library_scope_label": "focused",

        "library_perturbation_type_id": None,
        "library_perturbation_type_label": "inhibition",

        "library_manufacturer": "Marson lab",
        "library_lentiviral_generation": "2",
        "library_grnas_per_target": "2",
        "library_total_grnas": str(cur_data.adata.obs['guide_sequence'].str.split('|').explode().nunique()),
        "library_total_variants": None,

        "readout_dimensionality_id": None,
        "readout_dimensionality_label": "high-dimensional assay",

        "readout_type_id": None,
        "readout_type_label": "transcriptomic",

        "readout_technology_id": None,
        "readout_technology_label": "single-cell rna-seq",

        "method_name_id": None,
        "method_name_label": "Perturb-seq",

        "method_uri": None,

        "sequencing_library_kit_id": None,
        "sequencing_library_kit_label": "GEM-X Flex Gene Expression Human n-plex kit",

        "sequencing_platform_id": None,
        "sequencing_platform_label": "Ultima Genomics UG100",

        "sequencing_strategy_id": None,
        "sequencing_strategy_label": "barcode sequencing",

        "software_counts_id": None,
        "software_counts_label": "CellRanger",

        "software_analysis_id": None,
        "software_analysis_label": "scanpy",

        "reference_genome_id": None,
        "reference_genome_label": "GRCh38",

        "license_label": "MIT License",
        "license_id": "SWO:9000074",

        "associated_datasets": json.dumps([
            {
                "dataset_accession": "Primary Human CD4+ T Cell Perturb-seq",
                "dataset_uri": "s3://genome-scale-tcell-perturb-seq/marson2025_data/D1_Rest.assigned_guide.h5ad",
                "dataset_description": "Cell expression profiles for cells from donor D1_CE0008162 under resting conditions.",
                "dataset_file_name": "D1_Rest.assigned_guide.h5ad.h5ad",
            }
        ])
    }
)

In [ ]:
cur_data.adata.obs

### Curate tissue information


In [ ]:
cur_data.standardize_ontology(
    input_column='tissue',
    column_type='term_name',
    ontology_type='tissue',
    overwrite=True
)

### Curate cell type information

In [ ]:
cur_data.standardize_ontology(
    input_column='cell_type_label',
    column_type='term_name',
    ontology_type='cell_type',
    overwrite=True
)

### Curate disease information

In [ ]:
cur_data.standardize_ontology(
    input_column='disease_label',
    column_type='term_name',
    ontology_type='disease',
    overwrite=True
)

### Match schema column order

In [ ]:
cur_data.match_schema_columns(slot='obs')

### Validate obs metadata

In [ ]:
cur_data.validate_data(slot='obs', verbose=True)

# VAR slot curation

### Standardise genes

In [ ]:
cur_data.adata.var

In [ ]:
cur_data.standardize_genes(
    slot="var",
    input_column="gene_ids",
    input_column_type="ensembl_gene_id",
    remove_version=False,
    multiple_entries=False
)

In [ ]:
# replace gene symbols that are NOT ENSG with original provided gene symbol
cur_data.adata.var.loc[(cur_data.adata.var['gene_symbol'].isna()) &
 (~cur_data.adata.var['gene_name'].str.startswith('ENSG')), 'gene_symbol'] = cur_data.adata.var.loc[(cur_data.adata.var['gene_symbol'].isna()) &
 (~cur_data.adata.var['gene_name'].str.startswith('ENSG')), 'gene_name']

In [ ]:
cur_data.adata.var.loc[cur_data.adata.var['ensembl_gene_id'] == 'CUSTOM001_PuroR', 'ensembl_gene_id'] = None

### Validate var metadata

In [ ]:
cur_data.validate_data(slot='var')

# Save the dataset

In [ ]:
# Use 'object' to force standard Python strings (NumPy compatible)
cur_data.adata.obs.index = cur_data.adata.obs.index.astype("object")

# It is safer to do the same for the var index just in case
cur_data.adata.var.index = cur_data.adata.var.index.astype("object")

In [ ]:
import pandas as pd
import numpy as np

def convert_arrow_to_object(df):
    """
    Recursively converts all Arrow-backed string columns and categories 
    in a DataFrame to standard Python objects (numpy-compatible).
    """
    # 1. Fix Index
    if "arrow" in str(df.index.dtype).lower() or isinstance(df.index.dtype, pd.StringDtype):
        df.index = df.index.astype("object")
        
    for col in df.columns:
        col_dtype = df[col].dtype
        
        # 2. Fix Categorical Columns
        if isinstance(col_dtype, pd.CategoricalDtype):
            if "arrow" in str(col_dtype.categories.dtype).lower() or \
               isinstance(col_dtype.categories.dtype, pd.StringDtype):
                # Convert categories to object
                new_categories = df[col].cat.categories.astype("object")
                df[col] = df[col].cat.rename_categories(new_categories)
        
        # 3. Fix Standard String/Arrow Columns (e.g., sample_id)
        elif "arrow" in str(col_dtype).lower() or isinstance(col_dtype, pd.StringDtype):
            df[col] = df[col].astype("object")

# --- Apply the fix ---

print("Cleaning obs...")
convert_arrow_to_object(cur_data.adata.obs)

print("Cleaning var...")
convert_arrow_to_object(cur_data.adata.var)

In [ ]:
import pandas as pd
import numpy as np

def clean_adata_for_saving(adata):
    # 1. Fix Index
    adata.obs.index = adata.obs.index.astype("object")
    adata.var.index = adata.var.index.astype("object")
    
    for df in [adata.obs, adata.var]:
        for col in df.columns:
            # --- Fix 1: Convert Arrow/StringDtype to Object ---
            if "arrow" in str(df[col].dtype).lower() or isinstance(df[col].dtype, pd.StringDtype):
                df[col] = df[col].astype("object")
            
            # --- Fix 2: Handle Object columns with NaNs ---
            # If a column is object type and has NaNs, it will break h5py.
            # We convert these to 'category', which handles NaNs gracefully.
            if df[col].dtype == "object":
                if df[col].isnull().any():
                    print(f"Converting mixed-type column '{col}' to categorical...")
                    df[col] = df[col].astype("category")

            # --- Fix 3: Ensure Categories are Standard Objects ---
            # If we created a categorical, ensure its internal list of categories is not Arrow
            if isinstance(df[col].dtype, pd.CategoricalDtype):
                if df[col].cat.categories.dtype != "object":
                    new_cats = df[col].cat.categories.astype("object")
                    df[col] = df[col].cat.rename_categories(new_cats)

# Apply the comprehensive fix
clean_adata_for_saving(cur_data.adata)

In [ ]:
cur_data.adata.obs['dataset_id'].dtype

In [ ]:
cur_data.adata.obs['significant'].dtype

In [ ]:
cur_data.adata.obs['significant']

In [ ]:
cur_data.adata.obs

In [ ]:
cur_data.save_curated_data_h5ad()

In [ ]:
cur_data.save_curated_data_parquet(split_metadata=True, save_metadata_only=True)

# Upload to BigQuery

In [ ]:
# upload_parquet_to_bq(
#     parquet_path='../curated/parquet/zhu_2025_D1_rest_cl_curated_metadata.parquet',
#     bq_dataset_id='prj-ext-dev-pertcat-437314.perturb_seq',
#     bq_table_name='metadata',
#     key_columns=['dataset_id', 'sample_id'],
#     verbose=True
# )

# Upload to GC Storage

In [ ]:
!gcloud storage cp ../curated/h5ad/zhu_2025_D1_rest_cl_curated.h5ad gs://perturbation-catalogue-lake/perturbseq/curated/